# Seed Profiling
## Stage 1 of Dataset Discovery Pipeline

This notebook profiles any structured dataset and produces
`output/seed_signature.json` — consumed by Stage 2: Query Generation.

**Supported input formats:** CSV, TSV, JSON, Excel (.xlsx/.xls), Parquet, SQLite

> Run all cells top-to-bottom: **`Runtime → Run all`**


## 1. Setup

In [ ]:
import sys, os, subprocess
from pathlib import Path

# Project root is one level up from notebooks/
PROJECT_ROOT = Path("..").resolve()
SRC_PATH     = PROJECT_ROOT / "src"
OUTPUT_PATH  = PROJECT_ROOT / "output" / "seed_signature.json"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print(f"Project root : {PROJECT_ROOT}")
print(f"src/         : {SRC_PATH}  (exists: {SRC_PATH.exists()})")
print(f"output/      : {OUTPUT_PATH.parent}  (exists: {OUTPUT_PATH.parent.exists()})")


In [ ]:
# Install dependencies from requirements.txt
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r",
                       str(PROJECT_ROOT / "requirements.txt"), "-q"])
print("Dependencies ready.")


In [ ]:
from profiler import profile, save_signature, SUPPORTED_FORMATS
import pandas as pd
import json
print("Imports OK.")
print(f"Supported formats: {', '.join(SUPPORTED_FORMATS.keys())}")


## 2. Choose Your Seed Dataset

Point `SEED_PATH` at any supported file inside the `data/` folder.
Examples:
```python
SEED_PATH = PROJECT_ROOT / "data" / "seed.csv"
SEED_PATH = PROJECT_ROOT / "data" / "seed.json"
SEED_PATH = PROJECT_ROOT / "data" / "seed.xlsx"
SEED_PATH = PROJECT_ROOT / "data" / "seed.db"
```


In [ ]:
# ── Set your seed file here ──────────────────────────────────────
SEED_PATH = PROJECT_ROOT / "data" / "seed.csv"
# ─────────────────────────────────────────────────────────────────

print(f"Seed file : {SEED_PATH}")
print(f"Exists    : {SEED_PATH.exists()}")
print(f"Format    : {SEED_PATH.suffix.lower()}")


## 3. Load Preview

In [ ]:
ext = SEED_PATH.suffix.lower()
if ext == ".csv":
    _preview = pd.read_csv(SEED_PATH)
elif ext == ".tsv":
    _preview = pd.read_csv(SEED_PATH, sep="\t")
elif ext == ".json":
    _preview = pd.read_json(SEED_PATH)
elif ext in (".xlsx", ".xls"):
    _preview = pd.read_excel(SEED_PATH)
elif ext == ".parquet":
    _preview = pd.read_parquet(SEED_PATH)
elif ext in (".db", ".sqlite", ".sqlite3"):
    import sqlite3
    con = sqlite3.connect(SEED_PATH)
    tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", con)
    _preview = pd.read_sql(f"SELECT * FROM [{tables.iloc[0,0]}]", con)
    con.close()

print(f"Shape : {_preview.shape[0]:,} rows x {_preview.shape[1]} columns")
_preview.head()


In [ ]:
pd.DataFrame({
    "dtype":      _preview.dtypes.astype(str),
    "null_count": _preview.isna().sum(),
    "null_%":     (_preview.isna().mean() * 100).round(2),
    "unique":     _preview.nunique(),
})


## 4. Run the Profiler

In [ ]:
sig = profile(SEED_PATH)
save_signature(sig, OUTPUT_PATH)

print(f"Dataset  : {sig['dataset_name']}")
print(f"Format   : {sig['file_format']}")
print(f"Rows     : {sig['n_rows']:,}")
print(f"Columns  : {sig['n_columns']}")
print(f"Concepts : {sig['inferred_concepts']}")
print(f"Output   : {OUTPUT_PATH}")


## 5. Column-Level Profiles

In [ ]:
col_df = pd.DataFrame([
    {
        "column":      c["name"],
        "dtype":       c["raw_dtype"],
        "semantic":    c["semantic_type"],
        "null_%":      f"{c['null_ratio']*100:.1f}" if c['null_ratio'] is not None else "-",
        "unique_%":    f"{c['unique_ratio']*100:.1f}" if c['unique_ratio'] is not None else "-",
        "identifier?": "✓" if c["is_candidate_identifier"] else "",
        "entity?":     "✓" if c["is_entity_like"] else "",
        "query?":      "✓" if c["is_useful_for_querying"] else "",
        "join?":       "✓" if c["is_useful_for_joining"] else "",
    }
    for c in sig["columns"]
])
col_df


## 6. Global Extracted Signals

In [ ]:
print("── Candidate identifiers ──────────────────")
for c in sig["candidate_identifiers"]:
    print(f"  {c}")

print("\n── Candidate composite keys ───────────────")
for pair in sig["candidate_composite_keys"]:
    print(f"  {' + '.join(pair)}")
if not sig["candidate_composite_keys"]:
    print("  none found")

print("\n── Entity-like columns ────────────────────")
for c in sig["entity_like_columns"]:
    print(f"  {c}")

print("\n── Important attributes ───────────────────")
for c in sig["important_attributes"]:
    print(f"  {c}")


## 7. Quality Profile

In [ ]:
qp = sig["quality_profile"]
print(f"Duplicate rows   : {qp['duplicate_row_count']}")
print(f"High-missingness : {qp['high_missingness_columns'] or 'none'}")

if qp["outlier_summary"]:
    print("\nOutliers (IQR 1.5x):")
    for col, info in qp["outlier_summary"].items():
        print(f"  {col:30s}  {info['outlier_count']} outlier rows")
else:
    print("\nNo outliers detected.")


## 8. Retrieval-Oriented Summary

In [ ]:
print("── Salient column names ────────────────────────────────────")
print("  " + ", ".join(sig["salient_column_names"]))

print("\n── Entity value samples ────────────────────────────────────")
for col, vals in sig["entity_value_samples"].items():
    print(f"  {col}: {vals}")

print("\n── Inferred concepts ───────────────────────────────────────")
print("  " + ", ".join(sig["inferred_concepts"]))


## 9. Full Signature JSON (preview)

In [ ]:
preview = {k: v for k, v in sig.items() if k != "columns"}
print(json.dumps(preview, indent=2, default=str))


In [ ]:
# First 3 column profiles in full
for cp in sig["columns"][:3]:
    print(json.dumps(cp, indent=2, default=str))
    print()
